# Regional GVA (output) → ITL2

Builds **real regional GVA per ITL2 region per year** and saves it to `clean/gva_itl2.csv`.

- **Source:** ONS *Regional gross value added (balanced) by industry*, Table 2b — **chained volume measures, £ million** (i.e. **real** GVA, comparable over time). Published directly at ITL2.
- **What it is:** economic **output** (not productivity). It's the "GVA per region" measure in the RQ4 variable list. Productivity (GVA *per hour worked*) is a separate ONS release we'll add later. Industry breakdown for **sector mix** is also in this file — a separate follow-on.
- **Nice and simple:** already at ITL2 (2025) — the 46 region codes match the spine exactly, so **no crosswalk and no aggregation** needed. Covers all 46 regions incl. Scotland & NI. Years 1998–2023.

Put the downloaded workbook in `raw/gva/`. Run top to bottom; edit only `BASE`.

In [ ]:
import re
from pathlib import Path
import pandas as pd

# --- the ONE thing to edit: your project root ---
BASE = Path("/Users/h.cantekin/Library/CloudStorage/OneDrive-LondonSchoolofEconomics/Desktop/regional-panel")

GVA_DIR = BASE / "raw" / "gva"          # put the ONS workbook here
SPINE   = BASE / "clean" / "geography_spine.csv"
POP     = BASE / "clean" / "population_itl2.csv"   # for the optional per-capita step
OUT     = BASE / "clean" / "gva_itl2.csv"
OUT_PC  = BASE / "clean" / "gva_per_capita_itl2.csv"
SHEET   = "Table 2b"                    # ITL2, chained volume (real), £m

books = sorted(GVA_DIR.glob("*.xlsx"))
if not books:
    raise SystemExit(f"No .xlsx in {GVA_DIR}. Put the ONS regional GVA workbook there.")
GVA_FILE = books[-1]
print("Using:", GVA_FILE.name)

## 1. Extract total real GVA per region

Read Table 2b, keep the `Total / All industries` row for each ITL2 region, and reshape the year columns into tidy rows.

In [ ]:
df = pd.read_excel(GVA_FILE, sheet_name=SHEET, skiprows=1)
df.columns = [str(c).strip() for c in df.columns]

tot = df[df["SIC07 code"].astype(str).str.strip() == "Total"].copy()
tot = tot[tot["ITL code"].astype(str).str.startswith("TL")]      # ITL2 region rows only

year_cols = [c for c in tot.columns if re.fullmatch(r"\d{4}", str(c))]
gva = (tot.melt(id_vars=["ITL code", "Region name"], value_vars=year_cols,
                var_name="year", value_name="gva_real_gbp_m")
          .rename(columns={"ITL code": "itl2_code", "Region name": "itl2_name"}))
gva["year"] = gva["year"].astype(int)
gva["gva_real_gbp_m"] = pd.to_numeric(gva["gva_real_gbp_m"], errors="coerce")
gva = gva.sort_values(["year", "itl2_code"]).reset_index(drop=True)

print(f"{len(gva)} rows | {gva.itl2_code.nunique()} regions | years {gva.year.min()}-{gva.year.max()}")
gva.head()

## 2. Check the ITL2 codes match the spine, then save

No crosswalk is needed (these are ITL codes, not LA codes), but we confirm every region code exists in the spine so nothing is silently out of step with the rest of the panel.

In [ ]:
spine = pd.read_csv(SPINE)
spine_codes = set(spine.itl2_code.unique())
gva_codes = set(gva.itl2_code.unique())

missing_from_spine = gva_codes - spine_codes
missing_from_gva   = spine_codes - gva_codes
print(f"Spine regions: {len(spine_codes)} | GVA regions: {len(gva_codes)}")
print("GVA codes not in spine:", missing_from_spine or "none")
print("Spine codes not in GVA:", missing_from_gva or "none")

OUT.parent.mkdir(parents=True, exist_ok=True)
gva.to_csv(OUT, index=False)
print(f"\nSaved -> {OUT}")

## 3. (Optional) GVA per capita

Divide real GVA by population (from your population indicator) to get a rough output-per-person measure — closer to a productivity proxy than raw output. Joins on region and year where both exist.

In [ ]:
if POP.exists():
    pop = pd.read_csv(POP)[["itl2_code", "year", "population"]]
    pc = gva.merge(pop, on=["itl2_code", "year"], how="inner")
    pc["gva_per_capita_gbp"] = pc["gva_real_gbp_m"] * 1e6 / pc["population"]
    pc = pc[["itl2_code", "itl2_name", "year", "gva_real_gbp_m", "population", "gva_per_capita_gbp"]]
    pc.to_csv(OUT_PC, index=False)
    print(f"GVA per capita built for {pc.year.min()}-{pc.year.max()} -> {OUT_PC}")
    print("\n2023 top 5 by GVA per capita:")
    print(pc[pc.year==2023].nlargest(5, "gva_per_capita_gbp")[["itl2_name","gva_per_capita_gbp"]].to_string(index=False))
else:
    print("population_itl2.csv not found -- skipping per-capita (build population first).")

## 4. Coverage & sanity check

All 46 regions should be present (this source covers Scotland & NI). The 2023 ranking should put Inner London on top and rural/peripheral regions at the bottom.

In [ ]:
print(f"Coverage: {gva.itl2_code.nunique()} of {len(spine_codes)} regions x "
      f"{gva.year.nunique()} years ({gva.year.min()}-{gva.year.max()})")
print("\n2023 highest / lowest real GVA (£m):")
y = gva[gva.year == 2023]
print(pd.concat([y.nlargest(3, "gva_real_gbp_m"), y.nsmallest(3, "gva_real_gbp_m")])
        [["itl2_code","itl2_name","gva_real_gbp_m"]].to_string(index=False))